In [11]:
import numpy as np
import pandas as pd
import os
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.multivariate.manova import MANOVA


from bokeh.io import push_notebook, show, output_notebook
from bokeh.layouts import row
from bokeh.plotting import figure
from bokeh.palettes import Plasma256, viridis, Category20
from bokeh.models import ColumnDataSource, SingleIntervalTicker, BasicTickFormatter
from bokeh.layouts import gridplot
from bokeh.models import Legend, LegendItem


output_notebook()

Loading BokehJS ...

In [12]:
df = pd.read_csv("TOF_raw_data_df.csv")
print(df.shape)
print(df.head())

(4971, 138)
                                             Hash_id Labware_Name  \
0  09d08482b6f8631711a26aec6a1519c6ca412a866aec3c...     baseline   
1  09d08482b6f8631711a26aec6a1519c6ca412a866aec3c...     baseline   
2  09d08482b6f8631711a26aec6a1519c6ca412a866aec3c...     baseline   
3  09d08482b6f8631711a26aec6a1519c6ca412a866aec3c...     baseline   
4  09d08482b6f8631711a26aec6a1519c6ca412a866aec3c...     baseline   

          Stacker_SN Axis Platform_Position  Labware_Num_X  Labware_Num_Z  \
0  FSTA1020250401005    z            extend              0              0   
1  FSTA1020250401005    z            extend              0              0   
2  FSTA1020250401005    z            extend              0              0   
3  FSTA1020250401005    z            extend              0              0   
4  FSTA1020250401005    z            extend              0              0   

   Sample  Zone     Time  ...    119    120    121    122    123    124  \
0     1.0   0.0  0.92178  ...   31.

In [13]:
p1 = figure(width=1200,height=800, title="Z Axis Baseline with 6 STD envelope, Platform extended with no labware")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
for stacker in df['Stacker_SN'].unique():
    ys = df.query("Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1 & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = "Labware_Name=='baseline' & Axis=='z' & Platform_Position=='extend' & Labware_Num_X==0 & Zone==1"
# df_query += " & Stacker_SN in ['FSTA1020250409001','FSTA1020250408008','FSTA1020250408007','FSTA1020250403007','FSTA1020250407007','FSTA1020250403001','FSTA1020250407009','FSTA1020250403006','FSTA1020250402008','FSTA1020250402006','FSTA1020250402007','FSTA1020250401006','FSTA1020250403003','FSTA1020250407002','FSTA1020250407005','FSTA1020250407003','FSTA1020250403005','FSTA1020250402002']"

baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*6)
baseline_block = baseline_data.mean()-(baseline_data.std()*6)



p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 122


In [14]:
zone = 5
platform = 'retract'

zone = str(zone)
p1 = figure(width=1200,height=800, title=f"X Axis Baseline with 6 STD envelope, Platform {platform}ed, Zone {zone}")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
for stacker in df['Stacker_SN'].unique():
    ys = df.query(f"Labware_Name=='baseline' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone} & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = f"Labware_Name=='baseline' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*6)
baseline_block = baseline_data.mean()-(baseline_data.std()*6)


p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 125


In [15]:
zone = 6
platform = 'retract'

zone = str(zone)
p1 = figure(width=1200,height=800, title=f"X Axis Baseline with 6 STD envelope, Platform {platform}ed, Zone {zone}")
p1.xgrid.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.ticker = SingleIntervalTicker(interval=1, num_minor_ticks=0)
p1.xaxis.axis_label = "Bin"
p1.yaxis.axis_label = "# Photons"
p1.yaxis.formatter = BasicTickFormatter(use_scientific=False)

bin_range = [str(i) for i in range(1, 129)]

# Plot the raw data
color_count = 0
for stacker in df['Stacker_SN'].unique():
    ys = df.query(f"Labware_Name=='baseline' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone} & Stacker_SN==@stacker")[bin_range].values.tolist()
    numlines = len(ys)
    xs = [bin_range]*numlines

    color = Plasma256[color_count]
    p1.multi_line(xs = xs,
                    ys = ys,
                    line_color=color,
                    line_dash='solid',
                    legend_label=stacker)
    color_count += 10
    
# Create the baseline
df_query = f"Labware_Name=='baseline' & Axis=='x' & Platform_Position=='{platform}' & Labware_Num_X==0 & Zone=={zone}"
baseline_data = df.query(df_query)[bin_range]
print('Platform Home Baseline Samples: ' + str(len(baseline_data)))
baseline = baseline_data.mean()+(baseline_data.std()*6)
baseline_block = baseline_data.mean()-(baseline_data.std()*6)


p1.line(bin_range, baseline, line_color='green', line_width=2, line_dash='dashed')
p1.line(bin_range, baseline_block, line_color='red', line_width=2, line_dash='dashed')

p1.legend.click_policy="hide"
show(p1)

Platform Home Baseline Samples: 125
